# ਪਾਠ 18: ਕੁਪ੍ਰਤਿਆ ਯੋਗ ਰਸੀਦਾਂ ਨਾਲ ਏਆਈ ਏਜੰਟਾਂ ਨੂੰ ਸੁਰੱਖਿਅਤ ਕਰਨਾ

## ਹੱਥ-ਓਨ ਨੋਟਬੁੱਕ

ਇਹ ਨੋਟਬੁੱਕ ਚਾਰ ਅਭਿਆਸਾਂ ਵਿੱਚੋਂ ਲੰਘਦਾ ਹੈ:

1. ਇੱਕ ਏਜੰਟ ਟੂਲ ਕਾਲ ਲਈ ਆਪਣੀ ਪਹਿਲੀ ਰਸੀਦ **ਸਾਈਨ ਕਰੋ** ਅਤੇ ਇਸ ਦੀ ਪੁਸ਼ਟੀ ਕਰੋ।
2. ਰਸੀਦ **ਚੋਰੀ-ਚਪੇਟੀ ਕਰੋ** ਅਤੇ ਪੁਸ਼ਟੀ ਕਾਰਜ ਨਾਕਾਮ ਹੁੰਦਾ ਦੇਖੋ।
3. ਤਿੰਨ-ਰਸੀਦ ਕੰੜੀ **ਬਣਾਓ** ਅਤੇ ਕੰੜੀ ਦੀ ਸਚਾਈ ਦੀ ਪੁਸ਼ਟੀ ਕਰੋ।
4. ਮਾਇਕਰੋਸਾਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ ਟੂਲ ਕਾਲ ਨੂੰ **ਲਪੇਟੋ** ਤਾਂ ਜੋ ਹਰ ਕਦਮ 'ਤੇ ਇੱਕ ਰਸੀਦ ਜਾਰੀ ਹੋਵੇ।

ਸਾਰੇ ਕ੍ਰਿਪਟੋਗ੍ਰਾਫਿਕ ਪ੍ਰਿਮੀਟਿਵ ਲਾਇਬ੍ਰਰੀਆਂ ਤੋਂ ਆਏ ਹਨ (Ed25519 ਲਈ `pynacl`, RFC 8785 canonical JSON ਲਈ `jcs`, SHA-256 ਲਈ Python ਸਟੈਂਡਰਡ ਲਾਇਬ੍ਰੇਰੀ `hashlib`)। ਰਸੀਦ ਲੌਜਿਕ ਖੁਦ ਸੀਧਾ ਸਧਾਰਨ ਪਾਇਥਨ ਹੈ ਜੋ ਤੁਸੀਂ ਪੜ੍ਹ ਅਤੇ ਤਬਦੀਲ ਕਰ ਸਕਦੇ ਹੋ।

ਸੈੱਲਾਂ ਨੂੰ ਕ੍ਰਮ ਵਿੱਚ ਦੌੜਾਓ। ਹਰ ਹਿੱਸਾ ਛੋਟਾ ਅਤੇ ਖੁਦ-ਮੁਖਤਾਰ ਹੈ।


## ਸੈਟਅੱਪ

ਦੋਹਾਂ ਡੀਪੈਂਡੈਂਸੀਜ਼ ਨੂੰ ਇੰਸਟਾਲ ਕਰੋ। ਦੋਹਾਂ ਦੇ ਲਾਇਸੰਸ ਪਰਮਿਸਿਵ ਹਨ (Apache-2.0 / MIT)।


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## ਸਹਾਇਕ ਯੂਟਿਲਿਟੀਜ਼

ਇਹ ਦੋ ਸਹਾਇਕ ਬੇਸ64url ਐਨਕੋਡਿੰਗ (ਪੈਡਿੰਗ ਦੇ ਬਿਨਾਂ) ਅਤੇ ਕਿਸੇ ਵੀ ਵਸਤੂਆਂ ਦੀ SHA-256 ਹੈਸ਼ਿੰਗ ਕਰਦੇ ਹਨ। ਇਹ ਨੋਟਬੁਕ ਦੇ ਬਾਕੀ ਹਿੱਸੇ ਨੂੰ ਰਸੀਦ ਲੌਜਿਕ 'ਤੇ ਧਿਆਨ ਕੇਂਦਰਿਤ ਰੱਖਦੇ ਹਨ।


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## ਭਾਗ 1: ਆਪਣੀ ਪਹਿਲੀ ਰਸੀਦ 'ਤੇ ਦਸਤਖਤ ਕਰੋ

ਸੋਚੋ ਕਿ ਸਾਡੇ **ਕੋਂਟੋਸੋ ਟ੍ਰੈਵਲ** ਦੇ ਏਜੰਟ ਨੇ ਇੱਕ ਗ੍ਰਾਹਕ ਲਈ ਸਿਡਨੀ ਤੋਂ ਲੋਸ ਐਂਜਲਸ ਦੀਆਂ ਉਡਾਣਾਂ ਖੋਜੀਆਂ ਹਨ। ਅਸੀਂ ਇਸ ਟੂਲ ਕਾਲ ਨੂੰ ਇੱਕ ਦਸਤਖਤਸ਼ੁਦਾ ਰਸੀਦ ਵਜੋਂ ਦਰਜ ਕਰਨਾ ਚਾਹੁੰਦੇ ਹਾਂ ਤਾਂ ਜੋ ਭਵਿੱਖ ਦੇ ਆਡੀਟਰ ਇਸਦੀ ਜਾਂਚ ਕਰ ਸਕਣ ਬਿਨਾਂ ਸਾਡੇ ਉੱਤੇ ਭਰੋਸਾ ਕੀਤੇ।

### ਕਦਮ 1.1: ਸਾਈਨਿੰਗ ਕੀ ਜਨਰੇਟ ਕਰੋ

ਪੈਦਾਵਾਰੀ ਸਥਿਤੀ ਵਿੱਚ, ਏਜੰਟ ਦੀ ਸਾਈਨਿੰਗ ਕੀ ਹਾਰਡਵੇਅਰ ਸੁਰੱਖਿਆ ਮੋਡੀਊਲ (HSM), ਅਜ਼ੂਰ ਕੀ ਵਾਲਟ, ਜਾਂ ਇਸ ਤਰ੍ਹਾਂ ਦੇ ਕਿਸੇ ਸੁਰੱਖਿਅਤ ਸਟੋਰ ਵਿੱਚ ਰਹਿੰਦੀ ਹੈ। ਇਸ ਪਾਠ ਲਈ ਅਸੀਂ ਯਾਦਗਾਰੀ ਵਿੱਚ ਇੱਕ ਨਵੀਂ ਕੁੰਜੀ ਬਣਾਉਂਦੇ ਹਾਂ।


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### ਕਦਮ 1.2: ਰਸੀਦ ਪੇਲੋਡ ਬਣਾਉਣਾ

ਪੇਲੋਡ ਵਿੱਚ ਉਹ ਸਭ ਕੁਝ ਸ਼ਾਮਲ ਹੁੰਦਾ ਹੈ ਜਿਸ ਦੀ ਰਸੀਦ ਪੁਸ਼ਟੀ ਕਰਨੀ ਹੈ: ਕਿਸਨੇ ਕਾਰਵਾਈ ਕੀਤੀ, ਕਿਹੜਾ ਟੂਲ, ਕਿਹੜੇ ਆਰਗੂਮੈਂਟਸ ਨਾਲ, ਕੀ ਵਾਪਸ ਆਇਆ, ਕਿਸ ਨੀਤੀ ਹੇਠ, ਅਤੇ ਕਦੋਂ। ਅਸੀਂ ਆਰਗੂਮੈਂਟਸ ਅਤੇ ਨਤੀਜੇ ਦਾ ਹੈਸ਼ ਕਰਦੇ ਹਾਂ ਨਾ ਕਿ ਉਹ ਸੀਧੇ ਸ਼ਾਮਲ ਕਰਦੇ ਹਾਂ ਤਾਂ ਜੋ ਰਸੀਦ ਸੰਵੇਦਨਸ਼ੀਲ ਸਮੱਗਰੀ ਨੂੰ ਲੀਕ ਨਾ ਕਰੇ।


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### ਕਦਮ 1.3: ਰਸੀਦ 'ਤੇ ਸਾਇਨ ਕਰਨਾ ਅਤੇ ਇਕੱਠਾ ਕਰਨਾ

ਤਿੰਨ ਕਦਮ:

1. JCS ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਪੇਲੋਡ ਨੂੰ ਕੈਨਾਨਿਕਲਾਈਜ਼ ਕਰੋ ਤਾਂ ਜੋ ਦੋ ਇੰਪਲੀਮੈਂਟੇਸ਼ਨਾਂ ਜੋ ਇੱਕੋ ਲਾਜ਼ਮੀ ਰਸੀਦ ਬਣਾਉਂਦੀਆਂ ਹਨ ਉਹ ਬਾਈਟ-ਇਕਸਾਰ ਬਾਈਟ ਬਣਾਉਣ।
2. SHA-256 ਨਾਲ ਕੈਨਾਨਿਕਲ ਬਾਈਟਾਂ ਦਾ ਹੈਸ਼ ਕਰੋ।
3. Ed25519 ਨਿੱਜੀ ਕੁੰਜੀ ਨਾਲ ਹੈਸ਼ 'ਤੇ ਸਾਇਨ ਕਰੋ।

ਫਿਰ ਸਾਈਨ ਕੀਤਾ ਗਿਆ ਹੁਂਦਾ ਹੈ ਅਸਲ ਪੇਲੋਡ ਨਾਲ ਜੁੜਕੇ ਅੰਤੀਮ ਰਸੀਦ ਤਿਆਰ ਕਰਨ ਲਈ।


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### ਕਦਮ 1.4: ਰਸੀਦ ਦੀ ਜਾਂਚ ਕਰੋ

ਜਾਂਚ ਪ੍ਰਕਿਰਿਆ ਨੂੰ ਉਲਟ ਦਿੰਦੀ ਹੈ। ਅਸੀਂ ਹਸਤਾਖਰ ਨੂੰ ਹਟਾਉਂਦੇ ਹਾਂ, ਮੂਲ ਹੈਸ਼ ਮੁੜ ਗਣਨਾ ਕਰਦੇ ਹਾਂ, ਅਤੇ ਹਸਤਾਖਰ ਨੂੰ ਰਸੀਦ ਵਿੱਚ ਪ੍ਰਕਾਸ਼ਤ ਜਨਤਕ ਚਾਬੀ ਨਾਲ ਮਿਲਾਉਂਦੇ ਹਾਂ।

ਇਸ ਜਾਂਚ ਨੂੰ ਕਰਨ ਵਾਲੇ ਆਡੀਟਰ ਨੂੰ ਸਿਰਫ਼ ਰਸੀਦ ਦੀ ਲੋੜ ਹੁੰਦੀ ਹੈ। ਕਿਸੇ ਸੇਵਾ ਨੂੰ ਸੱਦਾ ਦੇਣ ਦੀ, ਕਿਸੇ ਕੁੰਜੀ ਨਿਦੇਸ਼ਿਕਾ ਦੀ ਪੁੱਛਗਿੱਛ ਕਰਨ ਦੀ, ਜਾਂ ਕਿਸੇ ਭਰੋਸੇ ਦੀ ਲੋੜ ਨਹੀਂ ਹੁੰਦੀ।


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

ਤੁਹਾਨੂੰ `Receipt is valid: True` ਵੇਖਾਈ ਦੇਣਾ ਚਾਹੀਦਾ ਹੈ। ਏਜੰਟ ਨੇ ਆਪਣਾ ਪਹਿਲਾ ਗੁਪਤ ਲੇਖਾ ਸਾਈਨ ਕੀਤਾ ਹੋਇਆ ਆਡੀਟ ਰਿਕਾਰਡ ਤਿਆਰ ਕੀਤਾ ਹੈ।  


## ਭਾਗ 2: ਰਸੀਦ ਨਾਲ ਛੇੜਛਾੜ ਕਰੋ

ਰਸੀਦਾਂ ਦਾ ਮੁੱਖ ਮਕਸਦ ਇਹ ਹੈ ਕਿ ਉਹ ਛੇੜਛਾੜ-ਸਬੂਤਨੁਮਾ ਹੁੰਦੀਆਂ ਹਨ। ਆਓ ਇਸਦਾ ਪਰਮਾਣ ਦਈਏ।

ਅਸੀਂ ਰਸੀਦ ਦੇ ਸਿਰਫ਼ ਇਕ ਅੱਖਰ ਨੂੰ ਸੋਧਾਂਗੇ ਤੇ ਵੇਰੀਫਿਕੇਸ਼ਨ ਫੇਲ ਹੋਣ ਦੀ ਦੇਖ ਬਣਾਉਂਗੇ।


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### ਕੀ ਹੁਣੀ ਸੀ?

ਜਦੋਂ ਅਸੀਂ `policy_id` ਨੂੰ ਬਦਲਿਆ, ਤਾਂ canonical bytes ਬਦਲ ਗਏ। ਉਹ bytes ਦਾ SHA-256 ਹੈਸ਼ ਬਦਲ ਗਿਆ। ਸਿਗਨੇਚਰ (ਜੋ ਅਸਲ ਹੈਸ਼ 'ਤੇ ਸੀ) ਹੁਣ ਨਵੇਂ ਹੈਸ਼ ਨਾਲ ਮੇਲ ਨਹੀਂ ਖਾਂਦੀ। ਵੈਰੀਫਿਕੇਸ਼ਨ ਸਹੀ ਤੌਰ ਤੇ `False` ਵਾਪਸ ਕਰਦਾ ਹੈ।

ਕੋਈ ਵੀ ਰਸੀਦ ਦਾ ਫ਼ੀਲਡ ਬਦਲ ਕੇ ਵੀ ਤਸਦੀਕ ਨਹੀਂ ਹੋ ਸਕਦੀ, ਜੇਕਰ ਹਮਲਾਵਰ ਕੋਲ ਨਿੱਜੀ ਕੁੰਜੀ ਨਾ ਹੋਵੇ। ਜਦ ਤੱਕ ਨਿੱਜੀ ਕੁੰਜੀ ਕੁੰਜੀ ਖਜ਼ਾਨੇ ਵਿੱਚ ਹੈ ਅਤੇ ਸਰਵਜਨ ਕੁੰਜੀ ਪ੍ਰਕਾਸ਼ਿਤ ਹੈ, ਤਬ ਤੱਕ ਛੇੜਛਾੜ ਨੂੰ ਛੁਪਾਉਣਾ ਅਸੰਭਵ ਹੈ।

ਖ਼ੁਦ ਕੋਸ਼ਿਸ਼ ਕਰੋ: ਉੱਪਰ ਦਿੱਤੀ ਸੈੱਲ ਵਿੱਚ `tool_name` ਜਾਂ `agent_id` ਜਾਂ `timestamp` ਬਦਲੋ ਅਤੇ ਮੁੜ ਚਲਾਓ। ਹਰ ਇੱਕ ਬਦਲਾਅ ਇੱਕ ਗਲਤ ਰਸੀਦ ਦਾ ਨਤੀਜਾ ਦਿੰਦਾ ਹੈ।


## ਭਾਗ 3: ਚੇਨ ਰਸੀਦਾਂ ਨੂੰ ਇਕੱਠਾ ਜੋੜੋ

ਇੱਕ ਇਕੱਲੀ ਰਸੀਦ ਇੱਕ ਕਾਰਵਾਈ ਦੀ ਸੁਰੱਖਿਆ ਕਰਦੀ ਹੈ। ਜ਼ਿਆਦਾਤਰ ਏਜੰਟ ਕਈ ਕਾਰਵਾਈਆਂ ਕਰਦੇ ਹਨ। ਪੂਰੀ ਕ੍ਰਮ ਨੂੰ ਟੈਂਪਰ-ਸਬੂਤ ਬਣਾਉਣ ਲਈ, ਅਸੀਂ ਹਰ ਨਵੀਂ ਰਸੀਦ ਵਿੱਚ ਪਿਛਲੀ ਰਸੀਦ ਦਾ ਹੈਸ਼ ਸ਼ਾਮਲ ਕਰਕੇ ਹਰ ਰਸੀਦ ਨੂੰ ਪਿਛਲੀ ਰਸੀਦ ਨਾਲ ਜੋੜਦੇ ਹਾਂ।

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

ਜੇ ਕੋਈ ਰਸੀਦ ਹਟਾ ਦਿੰਦਾ ਹੈ ਜਾਂ ਉਸਦੀ ਕ੍ਰਮਬੱਧਤਾ ਬਦਲਦਾ ਹੈ, ਤਾਂ ਸੇਰ ਉਸੇ ਬਿੰਦੂ 'ਤੇ ਟੁੱਟ ਜਾਂਦਾ ਹੈ। ਕਿਸੇ ਵੀ ਬਾਅਦ ਦੀ ਰਸੀਦ ਦੀ ਪੁਸ਼ਟੀ ਅਸਫਲ ਹੋ ਜਾਂਦੀ ਹੈ ਕਿਉਂਕਿ ਉਸਦਾ `previous_receipt_hash` ਹੁਣ ਉਸਦੇ ਪਿਛਲੇ ਵਾਲੇ ਦਾ ਅਸਲੀ ਹੈਸ਼ ਨਾਲ ਮੈਚ ਨਹੀਂ ਕਰਦਾ।


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

ਹੁਣ ਲੜੀ ਨੂੰ ਮੱਧਲੇ ਰਸੀਦ ਨੂੰ ਛੇੜ ਛਾੜ ਕਰ ਕੇ ਤੋੜੋ ਅਤੇ ਦੁਬਾਰਾ ਪਰਖ ਕਰੋ। ਛੇੜ ਛਾੜ ਕੀਤੀ ਰਸੀਦ ਆਪਣੀ ਦਸਤਖ਼ਤ ਜਾਂਚ ਵਿਚ ਫੇਲ ਹੋ ਜਾਂਦੀ ਹੈ, ਅਤੇ ਅਗਲੀ ਰਸੀਦ ਆਪਣੀ ਲੜੀ-ਲਿੰਕ ਜਾਂਚ ਵਿਚ ਫੇਲ ਹੋ ਜਾਂਦੀ ਹੈ (ਕਿਉਂਕਿ ਇਸ ਦਾ `previous_receipt_hash` ਹੁਣ ਬਦਲ੍ਹੇ ਮੱਧਲੇ ਰਸੀਦ ਦੇ ਹੈਸ਼ ਨਾਲ ਮੈਚ ਨਹੀਂ ਕਰਦਾ)।  


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

ਰਸੀਦ 0 ਅਜੇ ਵੀ ਜਾਂਚਦੀ ਹੈ (ਇਸਨੂੰ ਬਦਲਿਆ ਨਹੀਂ ਗਿਆ ਅਤੇ ਇਸਦਾ ਕੋਈ ਪੂਰਵਰੂਪ ਨਹੀਂ ਹੈ ਜਿਸ 'ਤੇ ਇਹ ਨਿਰਭਰ ਕਰਦੀ ਹੋਵੇ)। ਰਸੀਦ 1 ਆਪਣੀ ਦਸਤਖ਼ਤ ਜਾਂਚ ਵਿੱਚ ਅਸਫਲ ਹੁੰਦੀ ਹੈ ਕਿਉਂਕਿ ਅਸੀਂ `tool_args_hash` ਨੂੰ ਬਦਲ ਦਿੱਤਾ। ਰਸੀਦ 2 ਆਪਣੀ ਚੇਨ-ਲਿੰਕ ਜਾਂਚ ਵਿੱਚ ਅਸਫਲ ਹੁੰਦੀ ਹੈ ਕਿਉਂਕਿ ਇਸਦਾ `previous_receipt_hash` ਮੂਲ (ਹੁਣ ਬਦਲੀ ਹੋਈ) ਰਸੀਦ 1 ਦੇ ਮੁਕਾਬਲੇ ਗਣਨਾ ਕੀਤਾ ਗਿਆ ਸੀ।  

ਜੇਕਰ ਇੱਕ ਹਮਲਾਵਰ ਬਦਲੀ ਗਈ ਰਸੀਦ 1 ਨੂੰ ਦੁਬਾਰਾ ਦਸਤਖ਼ਤ ਕਰਦਾ ਹੈ (ਜੋ ਉਹ ਪ੍ਰਾਈਵੇਟ ਕੀ ਦੇ ਬਗैर ਨਹੀਂ ਕਰ ਸਕਦਾ), ਤਾਂ ਵੀ ਰਸੀਦ 2 ਵਿੱਚ ਚੇਨ-ਲਿੰਕ ਦੀ ਗਲਤ ਮੈਲ ਵਾਲਾ ਅੰਸ਼ ਤਬਾਹੀ ਨੂੰ ਬਿਆਨ ਕਰੇਗਾ। ਬਦਲਾਅ ਨੂੰ ਛੁਪਾਉਣ ਲਈ, ਹਮਲਾਵਰ ਨੂੰ ਸੋਧ ਦੇ ਬਿੰਦੂ ਤੋਂ ਬਾਅਦ ਹਰ ਰਸੀਦ ਨੂੰ ਮੁੜ ਦਸਤਖ਼ਤ ਕਰਨਾ ਪਵੇਗਾ, ਜਿਸ ਲਈ ਪ੍ਰਾਈਵੇਟ ਕੀ ਦੀ ਮਾਲਕੀ ਲੋੜੀਂਦੀ ਹੈ।  


## ਭਾਗ 4: ਏਜੈਂਟ ਟੂਲ ਕਾਲ ਨੂੰ ਰਸੀਦ ਸਾਈਨਿੰਗ ਨਾਲ ਲਪੇਟੋ

ਇੱਕ ਅਸਲੀ ਡਿਪਲੋਇਮੈਂਟ ਵਿੱਚ, ਤੁਸੀਂ ਚਾਹੁੰਦੇ ਨਹੀਂ ਕਿ ਹਰ ਏਜੈਂਟ ਲੇਖਕ ਨੂੰ `make_receipt` ਕਾਲ ਕਰਨ ਦੀ ਯਾਦ ਰਹੇ। ਤੁਸੀਂ ਚਾਹੁੰਦੇ ਹੋ ਕਿ ਹਰ ਟੂਲ ਕਾਲ ਲਈ ਰਸੀਦ ਸਾਈਨਿੰਗ ਸਵੈਚਾਲਿਤ ਹੋ ਜਾਵੇ।

ਇਹ ਰਹੀ ਸਭ ਤੋਂ ਸਧਾਰਣ ਪੈਟਰਨ: ਇੱਕ ਲਿਪੇਟਣ ਵਾਰਗ ਜੋ ਕਿਸੇ ਵੀ ਕਾਲ ਕਰਨਯੋਗ ਟੂਲ ਫੰਕਸ਼ਨ ਨੂੰ ਲੈਂਦਾ ਹੈ ਅਤੇ ਉਸ ਦਾ ਇੱਕ ਰਸੀਦ ਜਾਰੀ ਕਰਨ ਵਾਲਾ ਵਰਜਨ ਵਾਪਸ ਕਰਦਾ ਹੈ। ਇਹ ਕਿਸੇ ਵੀ ਏਜੈਂਟ ਫਰੇਮਵਰਕ ਨਾਲ ਮੈਚ ਕਰਦਾ ਹੈ, ਜਿਸ ਵਿੱਚ Microsoft Agent Framework (`agent_framework.foundry`) ਵੀ ਸ਼ਾਮਲ ਹੈ।

ਜੇ ਤੁਹਾਡੇ ਕੋਲ Microsoft Foundry ਪ੍ਰੋਜੈਕਟ ਸੈੱਟਅਪ ਨਹੀਂ ਹੈ, ਤਾਂ ਹੇਠਾਂ ਦਿੱਤਾ ਗਿਆ ਸਥਾਨਕ ਮੌਕ ਅਜੇ ਵੀ ਇਸ ਪੈਟਰਨ ਨੂੰ ਦਿਖਾਉਂਦਾ ਹੈ।


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### ਮਾਇਕਰੋਸਾਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ ਨਾਲ ਇੰਟੀਗ੍ਰੇਟ ਕਰਨਾ

ਉੱਪਰ ਦਿੱਤਾ `ReceiptedTool` ਰੈਪਰ ਫਰੇਮਵਰਕ-ਅੱਗਿਆਨ ਹੈ। ਇਸ ਨੂੰ ਮਾਇਕਰੋਸਾਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ ਨਾਲ ਬਣੇ ਏਜੰਟ ਦੇ ਅੰਦਰ ਵਰਤਣ ਲਈ, ਰੈਪ ਕੀਤਾ ਗਿਆ ਫੰਕਸ਼ਨ ਇੱਕ ਟੂਲ ਵਜੋਂ ਰਜਿਸਟਰ ਕਰੋ। ਇੱਕ ਖਾਕਾ (ਤੁਸੀਂ ਮੌਕ ਨੂੰ ਇੱਕ ਅਸਲੀ ਮਾਇਕਰੋਸਾਫਟ ਫਾਉਂਡਰੀ ਟੂਲ ਰਜਿਸਟ੍ਰੇਸ਼ਨ ਨਾਲ ਬਦਲੋਂਗੇ):

```python
# ਇੰਟੀਗ੍ਰੇਸ਼ਨ ਸ਼ੇਪ ਦਿਖਾਉਂਦਾ ਪੀਸੂਕੋਡ।
# ਆਯਾਤ ਕਰੋ os
# agent_framework.foundry ਤੋਂ FoundryChatClient ਆਯਾਤ ਕਰੋ
# azure.identity ਤੋਂ AzureCliCredential ਆਯਾਤ ਕਰੋ
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="ਤੁਸੀਂ Contoso Travel ਦਾ ਏਜੰਟ ਹੋ ..."
#     tools=[receipted_lookup],   # ਰੈਪ ਕੀਤਾ ਹੋਇਆ ਟੂਲ, ਕੱਚੀ ਫੰਕਸ਼ਨ ਨਹੀਂ
# )
# response = agent.run("ਜੂਨ ਵਿੱਚ Sydney ਤੋਂ Los Angeles ਲਈ ਉਡਾਣਾਂ ਲੱਭੋ।")
#
# # ਚਲਾਉਣ ਦੇ ਬਾਅਦ, ਹਰ ਟੂਲ ਕਾਲ ਜਿਸਨੂੰ ਏਜੰਟ ਨੇ ਕੀਤਾ ਹੈ, ਉਸਦਾ ਇੱਕ ਦਸਤਖ਼ਤਸ਼ੁਦਾ ਰਸੀਦ ਹੁੰਦੀ ਹੈ:
# audit_chain = receipted_lookup.receipts
```

ਏਜੰਟ ਫਰੇਮਵਰਕ ਨੂੰ ਰਸੀਦਾਂ ਬਾਰੇ ਕਿਸੇ ਗੱਲ ਦੀ ਜਾਣਕਾਰੀ ਹੋਣੀ ਜਰੂਰੀ ਨਹੀਂ। ਰਸੀਦ ਸਾਈਨਿੰਗ ਟੂਲ ਦੇ ਆਲੇ-ਦੁਆਲੇ ਲਪੇਟੀ ਹੁੰਦੀ ਹੈ, ਫਰੇਮਵਰਕ ਵਿੱਚ ਨਹੀਂ। ਇਹੋ ਤਰੀਕੇ ਨਾਲ ਤੁਸੀਂ ਮੌਜੂਦਾ ਏਜੰਟ ਕੋਡ ਵਿੱਚ ਪ੍ਰੋਵੇਨੈਂਸ ਜੋੜ ਸਕਦੇ ਹੋ ਬਿਨਾਂ ਏਜੰਟ ਨੂੰ ਦੁਬਾਰਾ ਲਿਖੇ।


## ਸੰਖੇਪ ਅਤੇ ਵਧੇਰੇ ਚੁਣੌਤੀ

ਤੁਹਾਡੇ ਕੋਲ ਹੈ:

- ਇੱਕ Ed25519 ਕੁੰਜੀ ਜੋੜੀ ਤਿਆਰ ਕੀਤੀ।
- ਏਜੰਟ ਟੂਲ ਕਾਲ ਲਈ ਇੱਕ ਰਸੀਦ ਬਣਾਈ ਅਤੇ ਸਾਈਨ ਕੀਤੀ।
- ਸਿਰਫ ਸਰਵਜਨਿਕ ਕੁੰਜੀ ਦੀ ਵਰਤੋਂ ਕਰਦੇ ਹੋਏ ਆਫਲਾਈਨ ਰਸੀਦ ਦੀ ਪੁਸ਼ਟੀ ਕੀਤੀ।
- ਇੱਕ ਰਸੀਦ ਵਿੱਚ ਤਬਦੀਲੀ ਕੀਤੀ ਅਤੇ ਪੁਸ਼ਟੀ ਫੇਲ ਹੋਣ ਦਾ ਮੁਆਇਨਾ ਕੀਤਾ।
- ਤਿੰਨ ਰਸੀਦਾਂ ਦੀ ਇੱਕ ਹੈਸ਼-ਚੇਨ ਸਰਣੀ ਬਣਾਈ।
- ਚੇਨ ਦੇ ਵਿਚਕਾਰ ਹਿੱਸੇ ਨਾਲ ਛੇੜਛਾੜ ਕੀਤੀ ਅਤੇ ਦੋਹਾਂ ਦਸਤਖ਼ਤ ਨਾਕਾਮੀ ਅਤੇ ਚੇਨ-ਲਿੰਕ ਨਾਕਾਮੀ ਦੇਖੀ।
- ਸਵੈਚਾਲਿਤ ਰਸੀਦ ਸਾਈਨਿੰਗ ਨਾਲ ਇੱਕ ਟੂਲ ਫੰਕਸ਼ਨ ਨੂੰ ਲਿਪੇਟਿਆ।

**ਵਧੇਰੇ ਚੁਣੌਤੀ।** ਇੱਕ `request_id` ਖੇਤਰ (ਵੰਡੇ ਹੋਏ ਟਰੇਸਿੰਗ ਲਈ ਇੱਕ UUID) ਨਾਲ ਰਸੀਦ ਸਕੀਮਾ ਨੂੰ ਵਧਾਓ। `make_receipt` ਨੂੰ ਅਪਡੇਟ ਕਰੋ ਤਾਂ ਜੋ ਇਹ ਸ਼ਾਮਿਲ ਹੋਵੇ, ਅਤੇ ਪੁਸ਼ਟੀ ਕਰੋ ਕਿ ਰਸੀਦਾਂ ਅੰਤ ਤੱਕ ਸਹੀ ਸਾਬਤ ਹੁੰਦੀਆਂ ਹਨ। ਫਿਰ ਸਾਈਨ ਕਰਨ ਤੋਂ ਬਾਅਦ ਖੇਤਰ ਨੂੰ ਬਦਲੋ ਅਤੇ ਪੁਸ਼ਟੀ ਕਰੋ ਕਿ ਫੇਲ ਹੁੰਦੀ ਹੈ। ਇਸ ਨਾਲ ਤੁਹਾਨੂੰ ਇਹ ਸਮਝਣ ਵਿੱਚ ਮਦਦ ਮਿਲਦੀ ਹੈ ਕਿ ਕੀਤੇ ਗਏ ਹਰ ਬਾਈਟ ਦਾ ਦਸਤਖ਼ਤ ਵਿੱਚ ਕਿਵੇਂ ਯੋਗਦਾਨ ਹੁੰਦਾ ਹੈ।

**ਮਹੱਤਵਪੂਰਨ ਸੀਮਾ।** ਰਸੀਦਾਂ ਤਿੰਨ ਗੱਲਾਂ ਨੂੰ ਸਾਬਤ ਕਰਦੀਆਂ ਹਨ ਅਤੇ ਸਿਰਫ਼ ਤਿੰਨ ਗੱਲਾਂ: ਸੋਪਣਾ (ਇਸ ਕੁੰਜੀ ਨੇ ਇਹ ਸਮੱਗਰੀ ਸਾਈਨ ਕੀਤੀ), ਅਖੰਡਤਾ (ਸਾਈਨ ਕਰਨ ਤੋਂ ਬਾਅਦ ਸਮੱਗਰੀ ਬਦਲੀ ਨਹੀਂ ਗਈ), ਅਤੇ ਕ੍ਰਮ (ਇਹ ਰਸੀਦ ਉਸ ਰਸੀਦ ਤੋਂ ਬਾਅਦ ਆਈ)। ਇਹ ਸਾਬਤ ਨਹੀਂ ਕਰਦੀਆਂ ਕਿ ਏਜੰਟ ਦੀ ਕਾਰਵਾਈ ਸਹੀ ਸੀ, ਕਿ `policy_id` ਵਿੱਚ ਨਾਂਮਿਤ ਨੀਤੀ ਦਾ ਅਸਲ ਵਿੱਚ ਮੁਲਾਂਕਣ ਕੀਤਾ ਗਿਆ, ਜਾਂ ਕਿ ਏਜੰਟ ਨੇ ਹਰ ਨਿਯਮ ਦਾ ਪਾਲਣ ਕੀਤਾ। ਰਸੀਦ ਇੱਕ ਬੁਨਿਆਦ ਹਨ। ਸ਼ਾਸਨ ਉਹ ਪ੍ਰਣਾਲੀ ਹੈ ਜੋ ਤੁਸੀਂ ਇਸ ਦੇ ਉੱਤੇ ਬਣਾਉਂਦੇ ਹੋ।

ਉਸ ਸੀਮਾ ਨੂੰ ਧਿਆਨ ਵਿੱਚ ਰੱਖਦਿਆਂ ਪਾਠਕ੍ਰਮ README ਨੂੰ ਮੁੜ ਪੜ੍ਹੋ। ਰਸੀਦਾਂ ਨਾਲ ਸਬੰਧਤ ਸਭ ਤੋਂ ਆਮ ਗਲਤੀ ਟੀਮਾਂ ਵੱਲੋਂ ਇਹ ਸਮਝਣਾ ਹੈ ਕਿ "ਸਾਡੇ ਕੋਲ ਰਸੀਦਾਂ ਹਨ" ਦਾ ਮਤਲਬ "ਅਸੀਂ ਸ਼ਾਸਿਤ ਹਾਂ" ਹੈ। ਇਹ ਸੱਚ ਨਹੀਂ ਹੈ। ਰਸੀਦਾਂ ਏਜੰਟ ਦੇ ਵਰਤਾਰਾ ਨੂੰ ਆਡੀਟ ਕਰਨ ਯੋਗ ਬਨਾਉਂਦੀਆਂ ਹਨ। ਇਹ ਇਸਨੂੰ ਸਹੀ ਨਹੀਂ ਬਨਾਉਂਦੀਆਂ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
